Objective

Build a system that recommends pricing, forecasts demand, segments products, and optimizes profit.

In [1]:
#IMPORTS
# =========================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

# =========================================
# 1. LOAD DATA
# =========================================
def load_data(path):
    df = pd.read_csv(path)
    return df

2. Feature Engineering (Core Foundation)

Transforms raw transactional data into meaningful business variables like profit, demand score, and time-based features.

In [2]:
# =========================================
# 2. FEATURE ENGINEERING
# =========================================
def feature_engineering(df):
    df = df.copy()

    df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y')
    df['month'] = df['order_date'].dt.to_period('M')

    # Cost assumption (70%)
    df['cost'] = df['price'] * 0.7

    # Profit calculations
    df['profit_per_unit'] = df['discounted_price'] - df['cost']
    df['total_profit'] = df['profit_per_unit'] * df['quantity_sold']

    # Demand signal
    df['demand_score'] = df['quantity_sold'] * df['rating']

    return df


3. Elasticity Engine (Pricing Intelligence)

Estimates how demand responds to changes in discount and price using regression.

In [3]:
# =========================================
# 3. ELASTICITY MODEL
# =========================================
def elasticity_model(df):
    df = df.copy()

    df['log_qty'] = np.log1p(df['quantity_sold'])
    df['log_discount'] = np.log1p(df['discount_percent'])

    X = df[['log_discount', 'rating', 'review_count']]
    X = sm.add_constant(X)

    y = df['log_qty']

    model = sm.OLS(y, X).fit()

    elasticity = model.params['log_discount']

    return model, elasticity


4. Forecasting Engine

Uses time-series modeling (like ARIMA model) to predict future demand at product or category level.

In [4]:
# 4. FORECASTING ENGINE (ARIMA)
# =========================================
def generate_forecasts(df, steps=3):

    forecasts = []

    product_ids = df['product_id'].unique()[:20]  
    # limit to 20 products for performance (can increase later)

    for pid in product_ids:
        try:
            data = df[df['product_id'] == pid]
            ts = data.groupby('month')['quantity_sold'].sum()

            if len(ts) < 3:
                continue

            model = ARIMA(ts, order=(1,1,1))
            model_fit = model.fit()

            pred = model_fit.forecast(steps=steps)

            for i, val in enumerate(pred):
                forecasts.append({
                    'product_id': pid,
                    'forecast_period': f"t+{i+1}",
                    'forecast_quantity': float(val)
                })

        except:
            continue

    return pd.DataFrame(forecasts)

5. Segmentation Engine

Classifies products into groups such as Stars, Cash Cows, Question Marks, and Dead Stock based on sales and growth.

In [5]:
# =========================================
# 5. SEGMENTATION ENGINE
# =========================================
def segmentation(df):
    agg = df.groupby('product_id').agg({
        'quantity_sold':'sum',
        'total_profit':'sum'
    }).reset_index()

    agg['growth'] = agg['quantity_sold'].pct_change().fillna(0)

    median_sales = agg['quantity_sold'].median()

    def segment(row):
        if row['quantity_sold'] > median_sales:
            if row['growth'] > 0:
                return 'Star'
            return 'Cash Cow'
        else:
            if row['growth'] > 0:
                return 'Question Mark'
            return 'Dead Stock'

    agg['segment'] = agg.apply(segment, axis=1)

    return agg

6. Profit Optimization Engine

Analyzes revenue vs profit margins to identify products that generate sales but may not be profitable.

In [6]:
# =========================================
# 6. PROFIT ANALYSIS
# =========================================
def profit_analysis(df):
    summary = df.groupby('product_id').agg({
        'total_revenue':'sum',
        'total_profit':'sum',
        'discount_percent':'mean'
    }).reset_index()

    summary['profit_margin'] = summary['total_profit'] / summary['total_revenue']

    return summary


7. Simulation Engine (THE STAR FEATURE)

Simulates different pricing or discount scenarios to estimate their impact on sales, revenue, and profit.

In [7]:
# =========================================
# 7. SIMULATION ENGINE (IMPROVED USING ELASTICITY)
# =========================================
def simulate_discount(df, product_id, elasticity, discount_range=range(0,50,5)):
    data = df[df['product_id'] == product_id]

    results = []

    base_qty = data['quantity_sold'].mean()
    base_price = data['price'].mean()
    cost = data['cost'].mean()

    for d in discount_range:
        new_price = base_price * (1 - d/100)

        # Elasticity-based demand change
        demand_multiplier = (1 + elasticity * (d/100))
        new_qty = max(base_qty * demand_multiplier, 0)

        revenue = new_price * new_qty
        profit = (new_price - cost) * new_qty

        results.append({
            'discount': d,
            'expected_qty': new_qty,
            'revenue': revenue,
            'profit': profit
        })

    return pd.DataFrame(results)

8. Decision Engine (MOST IMPORTANT)

Combines outputs from all models to generate actionable business recommendations

In [8]:
# =========================================
# 8. DECISION ENGINE
# =========================================
def generate_recommendations(elasticity, profit_df, segment_df):

    merged = profit_df.merge(segment_df[['product_id', 'segment']], on='product_id')

    results = []

    for _, row in merged.iterrows():

        reduce_discounting = 0
        increase_marketing = 0
        consider_removing = 0
        comment = []

        # Rule 1: Low profit margin
        if row['profit_margin'] < 0.1:
            reduce_discounting = 1
            comment.append("Low profit margin")

        # Rule 2: Segmentation logic
        if row['segment'] == 'Dead Stock':
            consider_removing = 1
            comment.append("Low sales & negative growth")

        elif row['segment'] == 'Star':
            increase_marketing = 1
            comment.append("High sales & growing demand")

        elif row['segment'] == 'Cash Cow':
            comment.append("Stable high sales")

        elif row['segment'] == 'Question Mark':
            comment.append("Low sales but growing")

        # Rule 3: Elasticity logic (global insight)
        if elasticity < -1:
            comment.append("Highly price sensitive")
        else:
            comment.append("Low price sensitivity")

        results.append({
            'product_id': row['product_id'],
            'reduce_discounting': reduce_discounting,
            'increase_marketing': increase_marketing,
            'consider_removing': consider_removing,
            'comment': ", ".join(comment)
        })

    return pd.DataFrame(results)


main pipeline

In [9]:
# 9. MAIN PIPELINE
# =========================================

def run_pipeline(path):

    df = load_data(path)
    df = feature_engineering(df)

    model, elasticity = elasticity_model(df)

    segment_df = segmentation(df)
    profit_df = profit_analysis(df)

    recommendations_df = generate_recommendations(elasticity, profit_df, segment_df)

    return df, model, segment_df, profit_df, elasticity, recommendations_df

Export pdf

In [10]:
# =========================================

def export_to_excel(df, segment_df, profit_df, sim_df, recommendations_df, forecast_df, output_path="pricing_insights.xlsx"):
    
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

        df.to_excel(writer, sheet_name='Processed_Data', index=False)
        segment_df.to_excel(writer, sheet_name='Product_Segments', index=False)
        profit_df.to_excel(writer, sheet_name='Profit_Analysis', index=False)
        sim_df.to_excel(writer, sheet_name='Simulation', index=False)
        recommendations_df.to_excel(writer, sheet_name='Recommendations', index=False)
        forecast_df.to_excel(writer, sheet_name='Forecast', index=False)

        # README
        readme_text = [
            ["Sheet Name", "Description"],
            ["Processed_Data", "Cleaned dataset with engineered features"],
            ["Product_Segments", "Product categorization"],
            ["Profit_Analysis", "Revenue, profit and margins"],
            ["Simulation", "Impact of discount strategies"],
            ["Recommendations", "Business decisions"],
            [],
            ["Business Terms", ""],
            ["Star", "High sales and positive growth → invest more marketing"],
            ["Cash Cow", "High sales but low growth → maintain"],
            ["Question Mark", "Low sales but growing → potential"],
            ["Dead Stock", "Low sales & negative growth → remove"],
            [],
            ["Decision Columns", ""],
            ["reduce_discounting", "1 = reduce discounts"],
            ["increase_marketing", "1 = increase ads"],
            ["consider_removing", "1 = remove product"],
            ["Forecast", "Predicted future demand (next 3 periods) using ARIMA model"],
            ["forecast_quantity", "Expected units to be sold"],
        ]

        pd.DataFrame(readme_text).to_excel(
            writer, sheet_name='README', index=False, header=False
        )

        # ============================
        # 🎨 COLOR FORMATTING
        # ============================
        workbook = writer.book
        sheet = writer.sheets['Recommendations']

        green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
        red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
        yellow_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")

        # Get column indexes
        cols = {cell.value: idx+1 for idx, cell in enumerate(sheet[1])}

        for row in range(2, sheet.max_row + 1):

            # reduce_discounting
            cell = sheet.cell(row=row, column=cols['reduce_discounting'])
            cell.fill = green_fill if cell.value == 1 else red_fill

            # increase_marketing
            cell = sheet.cell(row=row, column=cols['increase_marketing'])
            cell.fill = green_fill if cell.value == 1 else red_fill

            # consider_removing (special highlight)
            cell = sheet.cell(row=row, column=cols['consider_removing'])
            if cell.value == 1:
                cell.fill = yellow_fill
            else:
                cell.fill = red_fill

        # Auto adjust column width (nice touch)
        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            for col in ws.columns:
                max_length = 0
                col_letter = get_column_letter(col[0].column)

                for cell in col:
                    try:
                        if cell.value:
                            max_length = max(max_length, len(str(cell.value)))
                    except:
                        pass

                ws.column_dimensions[col_letter].width = max_length + 2

    print(f"\n✅ Excel report with formatting saved as: {output_path}")

In [11]:
# =========================================
# 10. RUN
# =========================================
if __name__ == "__main__":
    path = "amazon_sales_dataset.csv"

    # Run pipeline
    df, model, segment_df, profit_df, elasticity, recommendations_df = run_pipeline(path)

    

    # Pick product
    sample_product = df['product_id'].iloc[0]
    #forcast
    forecast_df = generate_forecasts(df)
    # Run simulation USING REAL elasticity
    sim_df = simulate_discount(df, sample_product, elasticity)

    # ✅ CALL THE FUNCTION (THIS WAS MISSING)
    export_to_excel(
    df,
    segment_df,
    profit_df,
    sim_df,
    recommendations_df,
    forecast_df
)

C:\Users\Bidisha Shit\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
C:\Users\Bidisha Shit\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
C:\Users\Bidisha Shit\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\Bidisha Shit\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameter


✅ Excel report with formatting saved as: pricing_insights.xlsx
